<a href="https://colab.research.google.com/github/kiruthikassrevg/Test_aiml_repo/blob/main/SQL%20Practices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import sqlite3

# Create an in-memory database
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# Create the orders table
cur.execute("""
CREATE TABLE orders (
    order_id INTEGER,
    customer TEXT,
    restaurant TEXT,
    amount REAL,
    status TEXT
)
""")

# Create the restaurants table
cur.execute("""
CREATE TABLE restaurants (
    name TEXT,
    cuisine TEXT
)
""")

# Create the restaurant_tags table
cur.execute("""
CREATE TABLE restaurant_tags (
    restaurant TEXT,
    tag TEXT
)
""")

In [13]:
orders_data = [
    (101, "Anita", "Spice Route", 540, "delivered"),
    (102, "Ravi", "Biryani House", 380, "delivered"),
    (103, "Meena", "Dosa Corner", 260, "cancelled"),
    (104, "Anita", "Dosa Corner", 120, "delivered"),
    (105, "Karthik", "Spice Route", 910, "delivered"),
    (106, "Divya", "Biryani House", 450, "delivered"),
    (107, "Ravi", "Pizza Point", 690, "cancelled"),
    (108, "Suresh", "Dosa Corner", 300, "delivered"),
    (109, "Priya", "Burger Barn", 410, "delivered")
]
restaurants_data = [
    ("Spice Route", "North Indian"),
    ("Biryani House", "Biryani"),
    ("Dosa Corner", "South Indian"),
    ("Pizza Point", "Italian")
]
tags_data = [
    ("Spice Route", "Great for Groups"),
    ("Spice Route", "Spicy"),
    ("Biryani House", "Spicy"),
    ("Dosa Corner", "Vegetarian-Friendly"),
    ("Pizza Point", "Family-Friendly")
]

In [14]:
cur.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?, ?)",
    orders_data
)

cur.executemany(
    "INSERT INTO restaurants VALUES (?, ?)",
    restaurants_data
)

cur.executemany(
    "INSERT INTO restaurant_tags VALUES (?, ?)",
    tags_data
)

conn.commit()

In [18]:
for row in cur.execute("""
    SELECT *
    FROM orders
"""):
    print(row)

(101, 'Anita', 'Spice Route', 540.0, 'delivered')
(102, 'Ravi', 'Biryani House', 380.0, 'delivered')
(103, 'Meena', 'Dosa Corner', 260.0, 'cancelled')
(104, 'Anita', 'Dosa Corner', 120.0, 'delivered')
(105, 'Karthik', 'Spice Route', 910.0, 'delivered')
(106, 'Divya', 'Biryani House', 450.0, 'delivered')
(107, 'Ravi', 'Pizza Point', 690.0, 'cancelled')
(108, 'Suresh', 'Dosa Corner', 300.0, 'delivered')
(109, 'Priya', 'Burger Barn', 410.0, 'delivered')


In [16]:
for row in cur.execute("SELECT * FROM restaurants"):
    print(row)

('Spice Route', 'North Indian')
('Biryani House', 'Biryani')
('Dosa Corner', 'South Indian')
('Pizza Point', 'Italian')


In [17]:
for row in cur.execute("SELECT * FROM restaurant_tags"):
    print(row)

('Spice Route', 'Great for Groups')
('Spice Route', 'Spicy')
('Biryani House', 'Spicy')
('Dosa Corner', 'Vegetarian-Friendly')
('Pizza Point', 'Family-Friendly')


Imagine you are the operations manager of a food-delivery company. You need a report showing how much money was generated by each cuisine from delivered orders. You only want cuisines with at least two delivered orders. How can you get this information?

In [19]:
for row in cur.execute("""
    SELECT restaurants.cuisine,
           COUNT(*) AS delivered_orders,
           SUM(orders.amount) AS delivered_revenue
    FROM orders
    INNER JOIN restaurants
        ON orders.restaurant = restaurants.name
    WHERE orders.status = 'delivered'
    GROUP BY restaurants.cuisine
    HAVING COUNT(*) >= 2
    ORDER BY delivered_revenue DESC
"""):
    print(row)

('North Indian', 2, 1450.0)
('Biryani', 2, 830.0)
('South Indian', 2, 420.0)


A restaurant manager wants to know not only which cuisine each order belongs to, but also which labels apply to that restaurant. For example, Spice Route is North Indian, and it has the tags ‘Spicy’ and ‘Great for Groups’

In [24]:
for row in cur.execute("""
    SELECT orders.order_id,
           orders.customer,
           restaurants.cuisine,
           restaurant_tags.tag
    FROM orders
    INNER JOIN restaurants
        ON orders.restaurant = restaurants.name
    INNER JOIN restaurant_tags
        ON restaurants.name = restaurant_tags.restaurant
    WHERE orders.restaurant = 'Spice Route'
    ORDER BY orders.order_id, restaurant_tags.tag
"""):
    print(row)

(101, 'Anita', 'North Indian', 'Great for Groups')
(101, 'Anita', 'North Indian', 'Spicy')
(105, 'Karthik', 'North Indian', 'Great for Groups')
(105, 'Karthik', 'North Indian', 'Spicy')


In [28]:
for row in cur.execute("""
    SELECT orders.order_id,
           orders.restaurant,
           restaurants.cuisine
    FROM orders
    LEFT JOIN restaurants
        ON orders.restaurant = restaurants.name

    ORDER BY orders.order_id
"""):
    print(row)

(101, 'Spice Route', 'North Indian')
(102, 'Biryani House', 'Biryani')
(103, 'Dosa Corner', 'South Indian')
(104, 'Dosa Corner', 'South Indian')
(105, 'Spice Route', 'North Indian')
(106, 'Biryani House', 'Biryani')
(107, 'Pizza Point', 'Italian')
(108, 'Dosa Corner', 'South Indian')
(109, 'Burger Barn', None)


In [31]:
for row in cur.execute("""
    SELECT orders.order_id,
           orders.restaurant,
           restaurants.cuisine
    FROM orders
    LEFT JOIN restaurants
        ON orders.restaurant = restaurants.name
    WHERE restaurants.cuisine != 'Italian' or restaurants.cuisine IS NULL
    ORDER BY orders.order_id
"""):
    print(row)

(101, 'Spice Route', 'North Indian')
(102, 'Biryani House', 'Biryani')
(103, 'Dosa Corner', 'South Indian')
(104, 'Dosa Corner', 'South Indian')
(105, 'Spice Route', 'North Indian')
(106, 'Biryani House', 'Biryani')
(108, 'Dosa Corner', 'South Indian')
(109, 'Burger Barn', None)
